<a href="https://colab.research.google.com/github/csu-techhub/quantum-optimization-simulation/blob/main/Module4_Labs/Lab12.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Lab 12 — Complete H₂ VQE with Four-Qubit Jordan–Wigner Encoding
**Quantum Optimization and Simulation — VQE Laboratory Series**

Now we move from the reduced two-qubit example to the full four-spin-orbital H₂ model.

\[
q_0=\sigma_g\uparrow,\quad q_1=\sigma_u^*\uparrow,\quad
q_2=\sigma_g\downarrow,\quad q_3=\sigma_u^*\downarrow.
\]

The Hartree–Fock occupation is \(|1010\rangle_{q_0q_1q_2q_3}\). We retain one paired double excitation,
\(|1010\rangle\leftrightarrow|0101\rangle\).

**Suggested use:** guided lab. Most code is supplied.

> In this lab `EstimatorV2` is used for energy expectation values. We count **Estimator energy evaluations**. The primitive may internally implement an expectation value with more than one physical measurement circuit.


## Learning objectives
1. Generate the four-qubit H₂ Hamiltonian with PySCF and Qiskit Nature.
2. Prepare the Hartree–Fock occupation state.
3. Explain why the two canonical-HF single excitations are omitted in this simple H₂ example.
4. Map one paired double excitation to Pauli strings.
5. Build a one-parameter UCC-style ansatz.
6. Minimize the total molecular energy with COBYLA.
7. Count how many Estimator energy evaluations are requested.


In [ ]:
%pip -q install pylatexenc matplotlib
%pip -q install "qiskit~=2.5" "qiskit-aer~=0.17" \
    "qiskit-algorithms~=0.4" "qiskit-nature~=0.8" "pyscf~=2.8"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

from qiskit import QuantumCircuit, transpile
from qiskit.circuit import Parameter
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.quantum_info import Statevector
from qiskit.synthesis import LieTrotter
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import EstimatorV2

from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.operators import FermionicOp

np.set_printoptions(precision=6, suppress=True)

In [ ]:
SEED = 123
SHOTS = 4096

def q0_first(qiskit_bits):
    return qiskit_bits.replace(" ", "")[::-1]

ESTIMATOR_EXECUTIONS = 0

def reset_execution_counter():
    global ESTIMATOR_EXECUTIONS
    ESTIMATOR_EXECUTIONS = 0

estimator = EstimatorV2()

def run_estimator(qc, observable, precision=0.0):
    global ESTIMATOR_EXECUTIONS

    backend = AerSimulator()
    tqc = transpile(qc, backend, optimization_level=1)

    # Aer keeps four qubits here, but apply_layout makes the function robust
    # if a nontrivial layout is introduced.
    mapped_observable = observable.apply_layout(tqc.layout)

    pub = (tqc, mapped_observable)
    result = estimator.run([pub], precision=precision).result()

    ESTIMATOR_EXECUTIONS += 1
    return float(np.real(result[0].data.evs))


**NOTE:**

`EstimatorV2` computes expectation values such as \(\langle H
angle\); it does not return measurement counts.

The execution counter in this notebook counts how many **energy-evaluation requests** are sent to the Estimator. On real hardware, one expectation-value request can require multiple physical measurement circuits.


## Part A — Prepare the Hartree–Fock state

In [ ]:
def hartree_fock_circuit():
    qc = QuantumCircuit(4)
    qc.x(0)
    qc.x(2)
    return qc

hf = hartree_fock_circuit()
display(hf.draw("mpl"))

## Part B - Single Double excitation

###Why are the two single excitations omitted?

The spin-preserving single excitations are

$
a_1^\dagger a_0:\ |1010\rangle\rightarrow|0110\rangle,
$

$
a_3^\dagger a_2:\ |1010\rangle\rightarrow|1001\rangle.
$

For minimal-basis H₂ using **canonical Hartree–Fock orbitals**:

- Singles mainly represent orbital rotations that HF has already optimized, so their
  optimal amplitudes are zero.
- This is a special result for this simple canonical-HF H₂ example. Singles should
not be removed automatically for general molecules or references.

### Map the paired double excitation

The anti-Hermitian UCC generator is

$
A=
a_1^\dagger a_3^\dagger a_2a_0
-
a_2^\dagger a_0^\dagger a_1a_3.
$

The unitary is $U_D(\theta)=e^{\theta A}$. Define the Hermitian generator

$
G=iA,
$

so that

$
U_D(\theta)=e^{-i\theta G}.
$

In [ ]:
double_excitation = FermionicOp(
    {
        # bonding alpha/beta -> antibonding alpha/beta
        "+_1 +_3 -_2 -_0": 1.0,

        # de-excitation
        "+_0 +_2 -_3 -_1": -1.0,
    },
    num_spin_orbitals=4,
)

mapper = JordanWignerMapper()
jw_antihermitian = mapper.map(double_excitation).simplify()
double_generator = (1j * jw_antihermitian).simplify()

print("Anti-Hermitian A:")
print(jw_antihermitian)

print("\nHermitian G=iA:")
print(double_generator)

labels = double_generator.paulis.to_labels()
coeffs = double_generator.coeffs

print("\nNumber of Pauli strings:", len(labels))
for label, coeff in zip(labels, coeffs):
    print(f"{coeff:+.6f}  {label}")

## Part C — Build the one-parameter ansatz

The eight Pauli strings are not eight independent excitations. Together they
represent **one** fermionic paired double excitation, so they all share one
parameter $\theta$.

`PauliEvolutionGate` synthesizes $e^{-i\theta G}$. We request a one-step
first-order Lie–Trotter product formula to mirror the lecture's sequence of eight
Pauli rotations.

In [ ]:
theta = Parameter("theta")

ansatz = hartree_fock_circuit()
ansatz.append(
    PauliEvolutionGate(
        double_generator,
        time=theta,
        synthesis=LieTrotter(reps=1),
    ),
    range(4),
)

display(ansatz.draw("mpl", fold=120))

decomposed_ansatz = transpile(ansatz, basis_gates=['cx', 'rz', 'h', 's', 'sdg', 'x'])
display(decomposed_ansatz.draw("mpl", fold=120))

print("Variational parameters:", ansatz.parameters)

## Part D — Generate the four-qubit Hamiltonian

`PySCFDriver` computes molecular orbitals and electronic integrals. Qiskit Nature
builds the second-quantized Hamiltonian, and `JordanWignerMapper` converts it into

$
H_{\mathrm{qubit}}=\sum_j c_jP_j,
$

where each $P_j$ is a four-qubit Pauli string.

In [ ]:
bond_distance = 0.735  # Angstrom

driver = PySCFDriver(
    atom=f"H 0 0 0; H 0 0 {bond_distance}",
    basis="sto3g",
    charge=0,
    spin=0,
)

problem = driver.run()
fermionic_hamiltonian = problem.hamiltonian.second_q_op()

mapper = JordanWignerMapper()
qubit_hamiltonian = mapper.map(fermionic_hamiltonian).simplify()
nuclear_repulsion = problem.nuclear_repulsion_energy

print("Spatial orbitals:", problem.num_spatial_orbitals)
print("Spin orbitals:", problem.num_spin_orbitals)
print("Particles (alpha, beta):", problem.num_particles)
print("Nuclear repulsion:", nuclear_repulsion, "Hartree")
print("Number of Pauli terms:", len(qubit_hamiltonian))
print(qubit_hamiltonian)

H_matrix = qubit_hamiltonian.to_matrix()

**NOTE:** H_matrix is the Hamiltonian matrix of $H_2$, mapped via the JW transformation.
* It captures the total energy operators of the molecular system, including electron kinetic energy, electron-nuclear attraction, and electron-electron repulsion.
* It is a 16x16 complex matrix (16 dimensions, because STO-3G for $H_2$ yields 4 spin-orbitals/qubits).
* The row and column indices represent the electronic configuration states in decimal form.

**Expected outcome:** two spatial orbitals, four spin orbitals, two electrons, and
a `SparsePauliOp` containing \(I\), \(Z\), \(ZZ\), and four-qubit \(X/Y\) terms.

## Part E — Define and scan the total energy

In [ ]:
# This initializes an empty list, which acts as a logger to store the history
# of parameters/energies so we can plot a convergence graph later.
evaluation_history = []

def total_energy(theta_array):
    theta_value = float(np.atleast_1d(theta_array)[0])

    # Bind the current double-excitation parameter.
    bound = ansatz.assign_parameters({theta: theta_value})

    # Compute the expectation value of the electronic Hamiltonian.
    electronic_energy = run_estimator(
        bound,
        qubit_hamiltonian,
    )

    # adds the constant classical potential energy of the nuclei
    # repelling each other.
    molecular_energy = electronic_energy + nuclear_repulsion

    evaluation_history.append((theta_value, molecular_energy))
    return molecular_energy

In [ ]:
reset_execution_counter()

scan_angles = np.linspace(-0.6, 0.6, 41)
scan_energies = [total_energy([angle]) for angle in scan_angles]

plt.plot(scan_angles, scan_energies, "o-")
plt.xlabel("Double-excitation parameter theta")
plt.ylabel("Total energy (Hartree)")
plt.title("Four-qubit H2 VQE energy landscape")
plt.grid(True)
plt.show()

print("Estimator energy evaluations in scan:", ESTIMATOR_EXECUTIONS)

## Part F — Optimize with COBYLA

In [ ]:
evaluation_history.clear()
reset_execution_counter()

result = minimize(
    total_energy,
    x0=np.array([0.0]),
    method="COBYLA",
    options={"maxiter": 60, "rhobeg": 0.2, "tol": 1e-8},
)

theta_opt = result.x[0]
energy_opt = result.fun

print(result)
print("\nOptimal theta:", theta_opt)
print("VQE total energy:", energy_opt, "Hartree")
print("VQE total energy:", energy_opt * 27.2114, "eV")

history_array = np.array(evaluation_history)
plt.plot(history_array[:, 1], marker=".")
plt.xlabel("Energy evaluation")
plt.ylabel("Total energy (Hartree)")
plt.title("COBYLA convergence")
plt.grid(True)
plt.show()

print("Estimator energy evaluations in COBYLA:", ESTIMATOR_EXECUTIONS)

## Part G — Inspect the optimized two-configuration mixture

In the logical orbital order \(q_0q_1q_2q_3\), the ansatz stays in

\[
\mathrm{span}\{|1010\rangle,|0101\rangle\}.
\]

Qiskit displays statevector labels in the reverse order \(q_3q_2q_1q_0\), so the printed dictionary labels are reversed.


In [ ]:
optimized = ansatz.assign_parameters({theta: theta_opt})
optimized_state = Statevector.from_instruction(optimized)
probabilities = optimized_state.probabilities_dict()

print({
    label: p for label, p in probabilities.items()
    if p > 1e-8
})

# Qiskit display order is q3 q2 q1 q0.
p_hf = probabilities.get("0101", 0.0)       # physical q0q1q2q3 = 1010
p_double = probabilities.get("1010", 0.0)   # physical q0q1q2q3 = 0101

print("HF physical |1010> probability:", p_hf)
print("Double physical |0101> probability:", p_double)
print("Total:", p_hf + p_double)


### YOUR TURN 3

Why does the ansatz have only one variational parameter even though the
Jordan–Wigner generator contains eight Pauli strings?

<details>
<summary><b>Suggested answer</b></summary>

The eight strings are the qubit representation of one fermionic paired
double-excitation operator. They are coordinated pieces of one physical process,
not eight independently adjustable excitations.

</details>

## Optional — Use Qiskit's `VQE` wrapper

In [ ]:
from qiskit.primitives import StatevectorEstimator
from qiskit_algorithms.minimum_eigensolvers import VQE
from qiskit_algorithms.optimizers import COBYLA

vqe = VQE(
    estimator=StatevectorEstimator(),
    ansatz=ansatz,
    optimizer=COBYLA(maxiter=60),
    initial_point=[0.0],
)

vqe_result = vqe.compute_minimum_eigenvalue(qubit_hamiltonian)
vqe_total = float(np.real(vqe_result.eigenvalue)) + nuclear_repulsion

print("Qiskit VQE electronic energy:", vqe_result.eigenvalue)
print("Qiskit VQE total energy:", vqe_total)
print("Optimal point:", vqe_result.optimal_point)

## Optional — Preview a fake-device noise model

Lab 4 introduced the difference between ideal and noisy execution. This optional cell shows how the same idea can be attached to the four-qubit Estimator calculation.

For a fuller noise study, including more-shots discussion and optional hardware, see Lab 6.


In [ ]:
%pip -q install "qiskit-ibm-runtime~=0.47"

from qiskit_ibm_runtime.fake_provider import FakeVigoV2
from qiskit_aer.noise import NoiseModel

fake_backend = FakeVigoV2()
fake_noise = NoiseModel.from_backend(fake_backend)

noisy_backend = AerSimulator(noise_model=fake_noise)

noisy_estimator = EstimatorV2(
    options={
        "backend_options": {
            "noise_model": fake_noise,
            "seed_simulator": SEED,
        }
    }
)

def run_noisy_estimator(qc, observable):
    tqc = transpile(
        qc,
        backend=noisy_backend,
        basis_gates=fake_noise.basis_gates,
        optimization_level=1,
        seed_transpiler=SEED,
    )

    mapped_observable = observable.apply_layout(tqc.layout)

    result = noisy_estimator.run(
        [(tqc, mapped_observable)]
    ).result()

    return float(np.real(result[0].data.evs))

print("Fake-device noise model is ready.")
print("Use run_noisy_estimator(bound_circuit, qubit_hamiltonian) for a comparison.")


## Final reflection

1. Why is the four-qubit Jordan–Wigner model more direct than the reduced model?
2. Why does this four-qubit circuit still explore only a small subspace of the full
   $2^4=16$-dimensional Hilbert space?
3. When would removing single excitations be unsafe?